# 32. JSON payloads that changed shape

**Track:** pyspark | **Difficulty:** medium | **Tags:** json, schema drift, get_json_object, from_json, semi-structured data

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("032-json-payload-schema-drift")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "raw_events": """
UEFSMRUEFZAPFeIHTBXyARUAEgAAyAcIIU4ABQEAIg0IACMNCAAkDQgAJQ0IACYNCAAnDQgAKA0IACkNCAAqDQgAKw0IACwNCAAt
DQgALg0IAC8NCAAwDQgAMQ0IADINCAAzDQgANA0IADUNCAA2DQgANw0IADgNCAA5DQgAOg0IADsNCAA8DQgAPQ0IAD4NCAA/DQgA
QA0IAEENCABCDQgAQw0IAEQNCABFDQgARg0IAEcNCABIDQgASQ0IAEoNCABLDQgATA0IAE0NCABODQgATw0IAFANCABRDQgAUg0I
AFMNCABUDQgAVQ0IAFYNCABXDQgAWA0IAFkNCABaDQgAWw0IAFwNCABdDQgAXg0IAF8NCABgDQgAYQ0IAGINCABjDQgAZA0IAGUN
CABmDQgAZw0IAGgNCABpDQgAag0IAGsNCABsDQgAbQ0IAG4NCABvDQgAcA0IAHENCAByDQgAcw0IAHQNCAB1DQgAdg0IAHcNCAB4
DQgAeQ0IAHoNCAB7DQgAfA0IAH0NCAB+DQgAfw0IAIANCACBDQgAgg0IAIMNCACEDQgAhQ0IAIYNCACHDQgAiA0IAIkNCACKDQgA
iw0IAIwNCACNDQgAjg0IAI8NCACQDQgAkQ0IAJINCACTDQgAlA0IAJUNCACWDQgAlw0IPJhOAAAAAAAAmU4AAAAAAAAVABXyARX4
ASwV8gEVEBUGFQYcGAiZTgAAAAAAABgIIU4AAAAAAAAWACgImU4AAAAAAAAYCCFOAAAAAAAAEREAAAB58HgDAAAA8gEBByGAgGBA
KBgOiIRiwWg4HpCIZEKpWC6YjGbD6Xg+oJBoRCqZTqiUasVquV6wmGxGq9luuJxux+v5fsCgcEgsGo/IpHLJbDqf0Kh0Sq1ar9is
dsvter/gsHhMLpvP6LR6zW673/C4fE6v2+94AAAAAAAAFQQVmmgV/CFMFfABFQASAACNNIhFAAAAeyJ1c2VyIjogeyJpZCI6IDQ1
MH0sICJwYXltZW50IgEYDGFtb3UFC2Q0Njk3LCAiY3VycmVuY3kiOiAiRVVSIn19MhVJAF8FRRAxMTIsIBk4CDUyOTo3ABRJTlIi
fSM+NgAENTQuNgAgMTY3MS41NH01OicAADgyJgAYMjAzOS40MzpgAAxBRUQiQjkABDM0NjoADDM0Ni5SmQAAMzpyAAQxMgFgNQcI
MjI5UjcAPqkABDIzMjcAEDkwMi43PkABJQlOiAEIMTUyZogBCDU5MTrxAAhVU0RKiAEANgGoHbgEMTY6NwAlKABDFe9ZBwAxYn0A
CDI1MwFWLgUCCX0AHxVHSQUINDE1Ms8BDDAzfSI6IwAANwFKFaAENDIlIgR9RxUmHZAAN2aSABAzMzkwLiFrLpQAJVkEfTYVSwmU
KboVchgyNzkwLjg4OhUBBTtOhQAIMjI0ZhcBDDAyLjFS3gEEfTg6hQAEMzgyGQEgIjQ0NTMuMzAiOocAJZw+wQAEMzMh+hnBCDQ0
N1p3AFJVAgQ5N2LBAAQ0MFbWAgR9MT6/AEFFFYIAMyXEQnoBAEoVNTnEBDM0ZtMCHCIxOTA0LjI5PgUBJYwEfUROTgAAOWLLAAwz
NjgwOk0BCUgASE5IAAAxZloCFDI0ODcuOD7WAVb0BAQyNDaaAQAwUlcCACAVgkmRNgoCEGZyZWUiQiQAADGldjVyDDEzMDFCbQMA
NzaVARA1NzB9NzprAAQyNiEVFUcYIjg1MS4zMVJxAT4BBQQzNzI6BQgzODE+bwOFAz4HBgA0IQMVcQQxOCUiLisCReNOagNuhQYM
NDkzLmUGLkoAKeBO9AIENDFmtQMEMzPFTUZJAD7LBQQzMjrLAAQxLsXjQjoAQjkAQT05AxAyOTcuOOEcLjkARVBkCAAAAG5vdCBq
c29uCQAAAFsxLCAyLCAzXSM+xwGFsBlTFDQyNy41MkKzAQQxMgFnGScEMzhBny53ACWzQuoBADQyJQIIMjc5QaIuNwAlaj7sBQAy
xdkZbgQ3MFK0AwQxM2aABAQ2M0FkLmoARR5StAMIMTQ1Zv0DEDgxMi4zUrYABH0vOkwBQcYZtQAwAfsdgQxudWxsQr0FBDQ2AVsV
NRgiNDUzOC4wQr0FJdRC+QWlYh08CDM5NEo8ACViPvkFBDI0MkoCFDQ1ODMuMgGLHbEJOkLTATacAQQ0NRLOCC43AAWtThwFBDE4
ZrIFDDIyMDVeaAE+5gQAMuUaFfkMMTc4NULhAAQxNyF/FSQUMjQ5Mi43YXYuqgBSGwEAMwUnFToQNDE1MS5WtwIANFUA2ZYENDhm
TAIUOTE1Ljc0RkcIBDQxMrkIBDE4IY0+YAMFrD6NAQQyNjKXAwgzOTKhuULjALlKHakEMDlm9QIMMzYyfVJ1BwQxMGbDARAiMTI1
N16BAlZNBQAzZi0BBDEwEmALLswARVlC9gkAMSWdNbAUMTE1OS43KmcIGfNq9QkANA6NCVI8BgQzM2bRBwgzNjdaVQJSHAIIMzM4
YnMBGDIxMzkuMX1C4gEAMzYcAgQ0NiXyLhYBhdgAHkWLUmcNHDEyMC41fX0TBSI2ng0AOUZwAAQyODLGBAgzNTUBEC5wAEWJAEZK
TgAANmbeABAzMDcxLmGRQkkAUpwCBDM5ZnQBEDE1MTN9Qu0NBDQ5QflZBRQ4OTEuMDH9QFkGCDMwNmaGARg4MjQuMDh9UjkAADRq
bwYYMjY5OS43OVY1AwQzMGYFDxAiMzk1NWI1A05GBQAzaoIIACIF91KFCQR9HDWzFiwKEnIINTIENDk+bAjBjAFKFSMIMTI3UpUE
GTk5ZQQxM2ZlARg0OTQyLjk2LvsBCZI6egUEODihFwBjKh8QEmQIACE6yAAyAgUYMTQ0MC40MlI5AQQ0N2ahAgw0NzYyWuYFQpMD
BDE1IQc1FwQzMCVKLqgAZSMAAgGoUlMEBDE5ZhACEDg3Ny4wVrsIucYpLwA0BSgViAA5QqYIpTA++gcANBLtDxU2FDM1MjAuMQHR
QsEA2d8JcBIGDRU5DDQwNTJW8BA+oQE+dgclxlk1CV0EMjCBaBVeEDc5OS4wUpAIACAhV4XICTkANBI9DBU5CDQ0NkasDAA1Mh0E
ECIyMDg2SnMDpUk+mAA6gAUQMTkxLjM+0QAl7z6eATauEQQyOKXIQmQBPtQJDCIxOTUydQMEMzJBEQ6tCS48ACVDPqABMkEBEDQx
MzEuQqERBao+EgIAOQHQNUEUIjI4Mi43VkQEPnwBADE2iQkMMjQyMEJBAQA1Ml4ABDQ2DiMIAaouzwAFl06WD2pNDwQxMgW4LkcA
RfBCgQAEMzYhPhXgDvAOAC4+YQMlUD7eAAA0NgsHDDQ3MTZSWAQIMzEzZlEGDDQzNy4SOBMuqQAF8FYbBGo4BQw2MTguQtYMCUtO
KAYENDVuRQcMMDguOGEWRpcAPgIDCCIzMzYIBgg0NTE+QAElenkC3QgANmaFABAzNjI0fUL8DQAxNq8BCDUwOB2SGVkIMjE3YnUB
GDE1MTAuMzcuhQmJLAHxVQkBf1p3Dz4iAwQyMg68CBU6EDQ1OTQuVqkSPtADOkQCGCIxNjE1LjVCmBJFgE66EAQzNGLmABQ0ODM3
LjQB00bEAQAjOiUFADISdhIVwRgxNzEzLjU1RtoGADcyTQcEMjZWugIMAAAAAFJyAgQ4NmKuAAwzMzE3Wm4BUmkMBDIyZlUDECIx
MzQ1SsEFaQwAHzr8AAA0MtQADDE1NzJCpgEWyhY1HxAiMzI0MEpgACWmJsAUWY0IMzYzYvkACDUzMkZDCDqFBwg0Mjk+EwXFjT7i
BQA0Zf8VqAg1MzNWggdOuRIAM2p1CAA06UVGSAI+CQMEIjE2ngYQNDY0OC4hYEI7ABUAFewBFfIBLBXyARUQFQYVBhw2AigCe30Y
ABERAAAAdvB1BwAAAFYBA/6MAQEHH4CAYEAoGA6IhGLBaDgekIhkQqlYLpiMZsPpeD6gkGhEKplOqJRqxWq5XrCYbEar2W64nG7H
6/l+wKBwSCwaj8ikcslsOp/QqHRKrVqv2Kx2y+16v+CweEwum8/otHrNbrvf8Lh8Tq/b7xUEGTw1ABgGc2NoZW1hFQQAFQQlAhgI
ZXZlbnRfaWQAFQwlAhgHcGF5bG9hZCUATBwAAAAW8gEZHBksJgAcFQQZNQAGEBkYCGV2ZW50X2lkFQIW8gEWqBIWgAsmjAgmCBwY
CJlOAAAAAAAAGAghTgAAAAAAABYAKAiZTgAAAAAAABgIIU4AAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYA8gEAAAAmABwV
DBk1AAYQGRgHcGF5bG9hZBUCFvIBFuhqFtAkJqYtJogLHDYCKAJ7fRgAEREAGSwVBBUAFQIAFQAVEBUCADwW2mAZBhkmAvABAAAA
FpB9FvIBJggW0C8AGRwYDEFSUk9XOnNjaGVtYRj4AS8vLy8vN0FBQUFBUUFBQUFBQUFLQUF3QUJnQUZBQWdBQ2dBQUFBQUJCQUFN
QUFBQUNBQUlBQUFBQkFBSUFBQUFCQUFBQUFJQUFBQkVBQUFBQkFBQUFOVC8vLzhBQUFFRkVBQUFBQndBQUFBRUFBQUFBQUFBQUFj
QUFBQndZWGxzYjJGa0FBUUFCQUFFQUFBQUVBQVVBQWdBQmdBSEFBd0FBQUFRQUJBQUFBQUFBQUVDRUFBQUFDUUFBQUFFQUFBQUFB
QUFBQWdBQUFCbGRtVnVkRjlwWkFBQUFBQUlBQXdBQ0FBSEFBZ0FBQUFBQUFBQlFBQUFBQT09ABggcGFycXVldC1jcHAtYXJyb3cg
dmVyc2lvbiAyNS4wLjEZLBwAABwAAAA7AgAAUEFSMQ==
""",
}

_data_dir = pathlib.Path("data") / "032-json-payload-schema-drift"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

raw_events = spark.read.parquet((_data_dir / "raw_events.parquet").resolve().as_posix())
raw_events.createOrReplaceTempView("raw_events")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'amount': '53274c5c4208c7dd7fb95b31e723f81fcce9c36735d473787d739a55fcb88b04',
                         'currency': '603dd0f759e5c7c01e49aaa752f39dbd8ca920456d307317b272dd65c9e38264',
                         'event_id': '4bc50b2db4152d7c84340927677a3116a39eabfe4a6aeb36e2f311dd03c22c4b',
                         'user_id': '4359aae3fe181cfe0d82e7a834d93477b758888b7933682231b5f467e712c9bc'},
 'columns': [['amount', 'double'],
             ['currency', 'string'],
             ['event_id', 'bigint'],
             ['user_id', 'bigint']],
 'fingerprint': '44d717298b215be36b78cf1e13c4b863a9944ad6fe9054fb4b639b275c00d515',
 'order_matters': False,
 'row_count': 121}

check = make_check(EXPECTED)

## The situation

A payments service writes one JSON document per event into a text column. Last year
the team restructured the payload: the user and the payment moved into nested
objects. Old and new events sit side by side in the same table.

The job that reads them was written for the old shape, with one fixed schema. It
still runs. It returns NULL for every field of every new-style event, and nobody
noticed until the revenue numbers halved.

## Input tables

### `raw_events`

One row per event.

| Column | Type | Description |
|---|---|---|
| `event_id` | BIGINT | Unique per event. |
| `payload` | STRING | A JSON document. Can be NULL, empty, or not valid JSON. |

The two payload shapes:

```
{"user_id": 12, "amount": 499.0, "currency": "INR"}
{"user": {"id": 12}, "payment": {"amount": 499.0, "currency": "INR"}}
```

In both shapes the amount is sometimes sent as a string (`"499.00"`), the user id is
sometimes a string (`"12"`), and the currency is sometimes left out.

## Requirement

Return every event with the three fields read from its payload.

- `user_id` comes from `user_id`, or from `user.id` in the new shape.
- `amount` comes from `amount`, or from `payment.amount`. A number sent as a string
  is still a number. Text that is not a number gives NULL.
- `currency` comes from `currency`, or from `payment.currency`. If the payload
  states no currency **and** the amount is not NULL, the currency is `INR`. If there
  is no currency and no amount, it is NULL.
- A field that is missing, or explicitly `null`, is NULL.
- A payload that is NULL, empty or not valid JSON gives NULL for all three fields.
- Every event appears. The job must not fail on a bad payload.

## Expected output

One row per event.

| Column | Type | Description |
|---|---|---|
| `event_id` | BIGINT | |
| `user_id` | BIGINT | |
| `amount` | DOUBLE | |
| `currency` | STRING | `INR` by default when there is an amount. |

In [ ]:
raw_events.show(5)

## Your answer

Use the DataFrame API. The tables are available as DataFrames: `raw_events`.

In [ ]:
result = None  # Replace None with your DataFrame.

In [ ]:
check(result)